# HISUI 指定領域スペクトルCSV作成

HISUIのGeoTIFF、同じベース名の `_B.csv`、`.txt` を読み込み、放射補正後のスペクトルをCSVに保存する

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

## 1. 入力ファイル・切り出し領域の設定

`hisui_file` にHISUIの `.tif` を指定。`_B.csv` と `.txt` は同じフォルダにあり、`.tif` と同じベース名で置かれている想定。

領域指定は `region_mode = "center"` なら中心座標と半幅、`region_mode = "bounds"` なら `y_min:y_max, x_min:x_max` で指定する。

In [ ]:
# Change this to your HISUI .tif file.
hisui_file = Path(
    r"C:\Users\yudon\OneDrive\デスクトップ\メタン\2025_HISUI_1_地獄の門\HSHL1G_N402E0583_20210803092156_20240106123056\HSHL1G_N402E0583_20210803092156_20240106123056.tif"
)

# Output CSV path. This avoids overwriting all_ir_spectra.csv by default.
output_csv = Path(r"D:\research\data\hisui_region_spectra.csv")

# "center" or "bounds"
region_mode = "center"

# Used when region_mode == "center".
center_y = 1410
center_x = 700
half_height = 25
half_width = 25

# Used when region_mode == "bounds". y_max/x_max are exclusive.
y_min = 1385
y_max = 1435
x_min = 675
x_max = 725

# "absolute" keeps original image coordinates. "relative" starts from 0 inside the region.
xy_mode = "absolute"

# Use None to save all 185 bands. Put numbers here to limit the wavelength range.
wave_min_nm = None
wave_max_nm = None

apply_radiometric_correction = True
drop_nodata_pixels = True
nodata_band = 10

## 2. HISUI読み込み・補正・CSV化の関数

In [ ]:
def _metadata_path(hisui_path, suffix):
    hisui_path = Path(hisui_path)
    return hisui_path.with_name(hisui_path.stem + suffix)


def _read_text_with_fallback(path):
    path = Path(path)
    for encoding in ("utf-8-sig", "utf-8", "cp932", "shift_jis"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    return path.read_text()


def read_bfile(hisui_path, n_bands=185):
    """Read HISUI *_B.csv band parameters."""
    bfile = _metadata_path(hisui_path, "_B.csv")
    if not bfile.exists():
        raise FileNotFoundError(f"_B.csv was not found: {bfile}")

    df = pd.read_csv(bfile)
    numeric = df.apply(pd.to_numeric, errors="coerce")

    if "CenterWavelengthNanometer" in df.columns:
        cols = [
            "CenterWavelengthNanometer",
            "FullWidthAtHalfMaximumNanometer",
            "SolarIrradianceWatt/Meter2/Micron",
            "ReflectanceMulti",
            "ReflectanceAdd",
        ]
        missing = [col for col in cols if col not in df.columns]
        if missing:
            raise ValueError(f"Required columns are missing in _B.csv: {missing}")
        param = numeric.loc[: n_bands - 1, cols].to_numpy(dtype=float)
    else:
        # Same assumption as command.ipynb: column 1 is band id, columns 2-6 are parameters.
        param = numeric.iloc[:n_bands, 1:6].to_numpy(dtype=float)

    if param.shape != (n_bands, 5):
        raise ValueError(f"Unexpected _B.csv shape: {param.shape}; expected {(n_bands, 5)}")
    if np.isnan(param[:, 0]).any():
        raise ValueError("Center wavelength contains NaN. Please check the _B.csv column layout.")
    return param


def read_tfile(hisui_path):
    """Read radiometric correction parameters from HISUI .txt metadata."""
    tfile = Path(hisui_path).with_suffix(".txt")
    if not tfile.exists():
        raise FileNotFoundError(f".txt metadata was not found: {tfile}")

    records = {}
    for line in _read_text_with_fallback(tfile).splitlines():
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        records[key.strip()] = value.strip()

    required = [
        "RadianceMultiVNIR",
        "RadianceAddVNIR",
        "RadianceMultiSWIR",
        "RadianceAddSWIR",
    ]
    missing = [key for key in required if key not in records]
    if missing:
        raise ValueError(f"Required keys are missing in .txt metadata: {missing}")

    return (
        float(records["RadianceMultiVNIR"]),
        float(records["RadianceMultiSWIR"]),
        float(records["RadianceAddVNIR"]),
        float(records["RadianceAddSWIR"]),
    )


def to_y_x_band(cube, n_bands=185):
    """Normalize tifffile/GDAL-like arrays to (y, x, band)."""
    cube = np.asarray(cube)
    if cube.ndim != 3:
        raise ValueError(f"Expected a 3D HISUI cube, got shape={cube.shape}.")

    if cube.shape[-1] == n_bands:
        return cube
    if cube.shape[0] == n_bands:
        return np.transpose(cube, (1, 2, 0))
    if cube.shape[1] == n_bands:
        return np.transpose(cube, (0, 2, 1))

    raise ValueError(f"Could not identify the 185-band axis: shape={cube.shape}")


def load_hisui_cube(hisui_path, n_bands=185):
    cube = tifffile.imread(hisui_path)
    cube = to_y_x_band(cube, n_bands=n_bands)
    return cube[..., :n_bands]


def apply_radiometric(cube, radmultivnir, radmultiswir, radaddvnir, radaddswir, nodata_band=10):
    """Apply VNIR/SWIR radiometric correction as in command.ipynb."""
    cube = np.asarray(cube)
    if cube.shape[-1] < 185:
        raise ValueError(f"At least 185 bands are required: shape={cube.shape}")

    valid = cube[:, :, nodata_band] != 0
    out = cube.astype(np.float32, copy=True)
    out[:, :, :58] = out[:, :, :58] * radmultivnir + radaddvnir
    out[:, :, 58:185] = out[:, :, 58:185] * radmultiswir + radaddswir
    out[~valid, :] = 0.0
    return out


def bounds_from_center(center_y, center_x, half_height, half_width):
    return (
        int(center_y - half_height),
        int(center_y + half_height),
        int(center_x - half_width),
        int(center_x + half_width),
    )


def clamp_bounds(y_min, y_max, x_min, x_max, image_shape):
    height, width = image_shape[:2]
    y0 = max(0, int(y_min))
    y1 = min(height, int(y_max))
    x0 = max(0, int(x_min))
    x1 = min(width, int(x_max))
    if y0 >= y1 or x0 >= x1:
        raise ValueError(f"Selected region is empty: y={y0}:{y1}, x={x0}:{x1}, image_shape={image_shape}")
    return y0, y1, x0, x1


def band_indices_for_wavelength_range(wavelengths, wave_min_nm=None, wave_max_nm=None):
    wavelengths = np.asarray(wavelengths, dtype=float)
    mask = np.ones(wavelengths.shape, dtype=bool)
    if wave_min_nm is not None:
        mask &= wavelengths >= float(wave_min_nm)
    if wave_max_nm is not None:
        mask &= wavelengths <= float(wave_max_nm)
    indices = np.where(mask)[0]
    if len(indices) == 0:
        raise ValueError(f"No bands in wavelength range: {wave_min_nm} - {wave_max_nm} nm")
    return indices


def spectra_columns(wavelengths):
    return [f"wave_{float(wave):.2f}nm" for wave in wavelengths]


def region_to_spectra_dataframe(
    cube,
    wavelengths,
    y_min,
    y_max,
    x_min,
    x_max,
    band_indices=None,
    xy_mode="absolute",
    drop_nodata_pixels=True,
    nodata_band=10,
):
    """Convert a rectangular region to y,x,wave_* rows like all_ir_spectra.csv."""
    wavelengths = np.asarray(wavelengths, dtype=float)
    if band_indices is None:
        band_indices = np.arange(len(wavelengths))
    else:
        band_indices = np.asarray(band_indices, dtype=int)

    region = cube[y_min:y_max, x_min:x_max, :][:, :, band_indices]
    height, width, n_selected_bands = region.shape

    yy, xx = np.meshgrid(np.arange(y_min, y_max), np.arange(x_min, x_max), indexing="ij")
    if xy_mode == "relative":
        yy = yy - y_min
        xx = xx - x_min
    elif xy_mode != "absolute":
        raise ValueError('xy_mode must be "absolute" or "relative".')

    values = region.reshape(height * width, n_selected_bands)
    yy = yy.reshape(-1)
    xx = xx.reshape(-1)

    if drop_nodata_pixels:
        valid = cube[y_min:y_max, x_min:x_max, nodata_band].reshape(-1) != 0
        valid &= ~np.all(np.isclose(values, 0.0, equal_nan=True), axis=1)
        yy = yy[valid]
        xx = xx[valid]
        values = values[valid]

    df = pd.DataFrame(values, columns=spectra_columns(wavelengths[band_indices]))
    df.insert(0, "x", xx.astype(int))
    df.insert(0, "y", yy.astype(int))
    return df


def make_rgb(cube, b=8, g=18, r=28, p_low=2, p_high=98):
    rgb = np.stack([cube[:, :, r], cube[:, :, g], cube[:, :, b]], axis=-1).astype(float)
    valid = np.isfinite(rgb) & (rgb != 0)
    if valid.any():
        low = np.nanpercentile(rgb[valid], p_low)
        high = np.nanpercentile(rgb[valid], p_high)
    else:
        low, high = 0.0, 1.0
    rgb = (rgb - low) / (high - low + 1e-12)
    return np.clip(rgb, 0.0, 1.0)

## 3. 読み込み・領域確認

In [ ]:
param = read_bfile(hisui_file)
wavelengths = param[:, 0]

cube = load_hisui_cube(hisui_file, n_bands=len(wavelengths))
print("cube shape (y, x, band):", cube.shape)
print("wavelength range:", f"{wavelengths.min():.2f} - {wavelengths.max():.2f} nm")

if apply_radiometric_correction:
    radmultivnir, radmultiswir, radaddvnir, radaddswir = read_tfile(hisui_file)
    cube = apply_radiometric(cube, radmultivnir, radmultiswir, radaddvnir, radaddswir, nodata_band=nodata_band)
    print("radiometric correction: applied")
else:
    print("radiometric correction: skipped")

if region_mode == "center":
    y0, y1, x0, x1 = bounds_from_center(center_y, center_x, half_height, half_width)
elif region_mode == "bounds":
    y0, y1, x0, x1 = y_min, y_max, x_min, x_max
else:
    raise ValueError('region_mode must be "center" or "bounds".')

y0, y1, x0, x1 = clamp_bounds(y0, y1, x0, x1, cube.shape)
print(f"selected region: y={y0}:{y1}, x={x0}:{x1} ({y1-y0} x {x1-x0} pixels)")

band_indices = band_indices_for_wavelength_range(wavelengths, wave_min_nm, wave_max_nm)
print(f"selected bands: {len(band_indices)}")

## 4. 画像プレビュー

矩形が正しい位置にあるか確認する。

In [ ]:
rgb = make_rgb(cube)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rgb)
axes[0].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="yellow", linewidth=2))
axes[0].set_title("Full image")
axes[0].set_axis_off()

axes[1].imshow(rgb[y0:y1, x0:x1, :])
axes[1].set_title("Selected region")
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

## 5. CSVとして保存


In [ ]:
spectra_df = region_to_spectra_dataframe(
    cube,
    wavelengths,
    y0,
    y1,
    x0,
    x1,
    band_indices=band_indices,
    xy_mode=xy_mode,
    drop_nodata_pixels=drop_nodata_pixels,
    nodata_band=nodata_band,
)

output_csv.parent.mkdir(parents=True, exist_ok=True)
spectra_df.to_csv(output_csv, index=False)

print(f"saved: {output_csv}")
print(f"rows: {len(spectra_df):,}")
print(f"columns: {len(spectra_df.columns):,}")
display(spectra_df.head())